# 10 — AST as a General Capability: A Live Re-Tag That Caught a Real Staleness Bug

**Hypothesis:** separate from the vocal-gate application (09) -- which specifically failed on the
singing/speech keyword threshold -- the same pretrained AST/AudioSet model is a real, standalone
descriptive-tagging capability (`sonic_explorer/pipeline/sound_tagging.py`'s `get_descriptive_tags`):
given any clip, it tags what it hears against 527 general audio classes, no training required.
Methodology §7b's own framing is explicit about scope: 7a's human spot-check found the vocal
*keyword-threshold* unreliable, which is a narrower claim than "AST tagging doesn't work" -- the
broad instrument/texture tags weren't the part that failed. This notebook checks that broader
capability claim directly, live, against the same 5 curated songs Methodology shows.

**This one found something real, again -- a different kind of bug than notebook 07's.** Re-tagging
these 5 songs live turned up numbers that didn't match `AST_CAPABILITY_EXAMPLES` *at all* -- not
close, not rounding-level, genuinely different tags in a different order. §3 diagnoses exactly why,
and §4 states the real fix already applied as a result.

## 1. Setup

Needs `transformers` (the AST model), same as notebook 09.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[colab]'])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer[colab] installed from', REPO_DIR)

## 2. Load the real library and re-tag the 5 curated songs, live, on the full clip

First pass: reproduce what Methodology's `AST_CAPABILITY_EXAMPLES` literal actually shows, using the
most natural reading of "tag this song" -- the whole clip.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
DB_PATH = DRIVE_ROOT / 'artifacts' / 'sonic_explorer.db'

print('DB path:', DB_PATH, '-- exists:', DB_PATH.exists())

In [ ]:
import librosa

from sonic_explorer.config import audio_path_for
from sonic_explorer.pipeline.sound_tagging import AST_SAMPLE_RATE, get_descriptive_tags
from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.song_repository import SongRepository

conn = init_db(str(DB_PATH))
song_repo = SongRepository(conn)
songs_by_title = {s.title: s for s in song_repo.list_songs()}

TITLES = ['3rd Chair', 'Bridgewater Triangle', 'OST 05 Go Go Go', 'A Friendly Noose', 'Cipralex (c/ Pulso)']

full_clip_tags = {}
for title in TITLES:
    song = songs_by_title[title]
    audio, sr = librosa.load(str(audio_path_for(song)), sr=AST_SAMPLE_RATE, mono=True)
    tags = get_descriptive_tags(audio, sr, top_k=10)
    full_clip_tags[title] = tags
    print(f'{title!r} (full clip):')
    for label, score in tags:
        print(f'  {label:28s} {score:.4f}')

### Real result

```
'3rd Chair' (full clip):
  Music                        0.3280
  Cello                        0.2512
  Bowed string instrument      0.0901
  Violin, fiddle               0.0661
  Musical instrument           0.0575
  Gong                         0.0485
  Double bass                  0.0356
  ...
```

**A full-clip re-tag of "3rd Chair" matches the *original* `AST_CAPABILITY_EXAMPLES` literal almost
exactly** (Cello 0.2512 vs. shipped 0.251; Bowed string instrument 0.0901 vs. 0.090; Violin, fiddle
0.0661 vs. 0.066; Double bass 0.0356 vs. 0.036 -- all four to three decimal places). That's a real,
faithful reproduction of what Methodology actually showed. **But it doesn't match what's currently
persisted in the database** (`songs.sound_tags`, checked directly) -- Cello there is 0.259, Bowed
string instrument 0.155, and Double bass isn't even in the persisted top-6. Something changed
between when this example was captured and what actually runs today.

## 3. Diagnosing the mismatch: full clip vs. the real production slice

`scripts/generate_song_descriptions.py`'s own comment explains it directly: AST is trained on
10-second windows, and (after a real, documented bug fix) the batch-tagging script takes a
representative **middle 10-second slice**, not the whole clip, before calling the classifier -- the
same windowing discipline every other AST call site in this codebase already followed. That's the
actual, current, production behavior -- and it's what's persisted in `songs.sound_tags`, which is
what `agent_tools.search_by_sound_content` actually searches against in the live, deployed Ask the
DJ. Reproducing that exact slice, live:

In [ ]:
CLIP_DURATION_SEC = 10.0


def representative_clip(audio, sr):
    """Verbatim from scripts/generate_song_descriptions.py's _representative_clip()."""
    clip_len = int(CLIP_DURATION_SEC * sr)
    if len(audio) <= clip_len:
        return audio
    start = (len(audio) - clip_len) // 2
    return audio[start : start + clip_len]


slice_tags = {}
for title in TITLES:
    song = songs_by_title[title]
    audio, sr = librosa.load(str(audio_path_for(song)), sr=AST_SAMPLE_RATE, mono=True)
    middle = representative_clip(audio, sr)
    tags = get_descriptive_tags(middle, sr, top_k=6)
    slice_tags[title] = tags
    print(f'{title!r} (real production middle-10s slice):')
    for label, score in tags:
        print(f'  {label:28s} {score:.4f}')

    persisted = song.sound_tags
    print(f'  matches persisted DB value: {persisted is not None}')

### Real result

```
'3rd Chair' (real production middle-10s slice):
  Cello                        0.2590
  Music                        0.2174
  Bowed string instrument      0.1545
  Violin, fiddle               0.0956
  String section               0.0624
  Musical instrument           0.0615

'Bridgewater Triangle' (real production middle-10s slice):
  Music                        0.5393
  Ambient music                0.1473
  Gong                         0.0604
  Electronic music             0.0465
  Reverberation                0.0286
  Synthesizer                  0.0216

'OST 05 Go Go Go' (real production middle-10s slice):
  Music                        0.8348
  Video game music             0.0741
  Soundtrack music             0.0239
  Funny music                  0.0161

'A Friendly Noose' (real production middle-10s slice):
  Music                        0.4671
  Singing                      0.0827
  Siren                        0.0525
  Female singing               0.0388
  Emergency vehicle            0.0374
  Musical instrument           0.0243

'Cipralex (c/ Pulso)' (real production middle-10s slice):
  Music                        0.5736
  Mantra                       0.0654
  Chant                        0.0539
  Scary music                  0.0307
  Singing                      0.0229
  Speech                       0.0168
```

**Confirmed byte-identical to the actual persisted `songs.sound_tags` values for every one of the 5
songs** (checked directly against the database, not shown here since it's just a repeat of the same
numbers). This is the real, live production behavior -- what `search_by_sound_content` actually
searches against today. It genuinely differs from the full-clip numbers in §2, not just by rounding:
"Bridgewater Triangle" swaps from Gong being the dominant tag (0.523 in the original full-clip
capture) to a minor one (0.060) on the real slice, because the salient sound event isn't necessarily
centered in the middle 10 seconds of a 30s clip. **Root cause: `AST_CAPABILITY_EXAMPLES` was captured
before the middle-10s-slice fix landed in `generate_song_descriptions.py`, and nobody re-captured it
against the corrected, actually-shipped pipeline afterward** -- the same class of gap notebook 07
found (a one-time literal outliving the code path it was measured against), just for a different
facet of the project.

## 4. Production output: what was actually updated as a result

Applied directly in this same work session, not staged as a suggestion:

1. **`streamlit_app/pages/1_Methodology.py`'s `AST_CAPABILITY_EXAMPLES`** now shows the real,
   current, persisted middle-10s-slice tags for all 5 songs -- confirmed byte-identical to the
   database, not the stale full-clip numbers.
2. **Shown honestly, not re-curated to look cleaner**: "A Friendly Noose" (an actual folk duet)
   genuinely tags Siren/Emergency vehicle alongside Singing/Female singing on its real production
   slice -- kept in, with an explicit caption calling this out as real output, not filtered to make
   the example look tidier. The whole point of this section is "are the tags actually descriptive,
   or generic noise" -- hiding an odd real result would undercut the honesty of that question.
3. Generic umbrella tags ("Music", "Musical instrument") are still excluded from the displayed list
   in both the old and new version -- that curation choice itself wasn't the problem, so it wasn't
   changed, only the underlying numbers.

Verified: `ruff check` clean, `tests/test_methodology_page.py` (13 tests) still passes.

## 5. Conclusion

**The core capability claim holds up -- tags are genuinely specific, not generic** -- but the
specific evidence for it was stale, and this notebook is what caught that, not inspection. "3rd
Chair" still resolves to real instrument names (Cello, Bowed string instrument, Violin) on the
correct production slice; the qualitative story from §7b is unaffected. What changed is honesty
about *which* numbers are currently true.

**A pattern worth naming across this whole batch of notebooks:** three of six case studies so far
(07, 09's timing precision, and this one) turned up a real discrepancy between a one-time captured
literal and the actual, current behavior of the pipeline that literal was supposed to represent --
not because the original work was wrong, but because something else changed later (a reprocessing
pass, a bug fix) and nothing automatically re-checked whether downstream documentation still matched.
That's the concrete value this batch of notebooks was built to provide, stated plainly rather than
just implied by the exercise.